# deeptool 퀵스타트

선형회귀를 예제로 `deeptool` 의 4가지 기능을 보여준다:
하이퍼파라미터 자동 저장, 셀 간 메서드 추가, 라이브 손실 곡선, 체크포인트.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

import deeptool as dt

## 1. 데이터 — `DataModule` 서브클래싱

In [ ]:
class SyntheticRegression(dt.DataModule):
    def __init__(self, n=200, batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        torch.manual_seed(0)
        self.X = torch.randn(n, 2)
        self.y = self.X @ torch.tensor([[2.0], [-3.4]]) + 4.2

    def get_dataloader(self, train):
        idx = slice(0, 160) if train else slice(160, None)
        return self.get_tensorloader((self.X, self.y), train, idx)


data = SyntheticRegression()
data.hparams

## 2. 모델 — `Module` 서브클래싱

`self.net` 만 정의하면 `forward` 는 자동으로 위임된다.

In [ ]:
class LinearRegression(dt.Module):
    def __init__(self, lr=0.03):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.LazyLinear(1)

## 3. 나중 셀에서 메서드 덧붙이기 — `@dt.add_to_class`

클래스 정의를 다시 실행할 필요가 없다.

In [ ]:
@dt.add_to_class(LinearRegression)
def loss(self, y_hat, y):
    return F.mse_loss(y_hat, y)


@dt.add_to_class(LinearRegression)
def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=self.lr)

## 4. 학습 — 손실 곡선이 이 셀 출력에 실시간으로 그려진다

In [ ]:
model = LinearRegression()
trainer = dt.Trainer(max_epochs=20)
trainer.fit(model, data)

In [ ]:
trainer.history["train_loss"][-1], trainer.history["val_loss"][-1]

## 5. 체크포인트

In [ ]:
trainer.save_checkpoint("linreg.pt")

restored = LinearRegression()
restored(data.X[:1])  # LazyLinear 실체화
meta = dt.Trainer.load_checkpoint("linreg.pt", restored)
meta